# 1. Validated kinase-domain ESM feature generation

This is a corrected **new-extraction workflow**, not a reconstruction of the
historical 1,003-row training feature matrix. Only mutated KD sequences are
embedded. Raw outputs contain two 1,280-dimensional vectors per scorable row;
PCA belongs to the separate training/artifact workflow in notebook 2.

The original `mutant_df.csv`, `wildtype_kinase_data.csv` and license are retained
unchanged. Do not substitute newly extracted values for historical training
features, and do not change experimental labels to match model outputs.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from workflow import validate_variants, ESMExtractor, generate_raw_features, sha256

SOURCE_DIR = Path(os.environ.get('KINOME_SOURCE_DIR', '.')).resolve()
OUTPUT_DIR = Path(os.environ.get('KINOME_OUTPUT_DIR', 'outputs')).resolve()
RUN_ESM = os.environ.get('KINOME_RUN_ESM', '0') == '1'
LIMIT = int(os.environ.get('KINOME_ESM_LIMIT', '0'))
LOCAL_MODEL_ONLY = os.environ.get('KINOME_LOCAL_MODEL_ONLY', '0') == '1'
if LIMIT < 0:
    raise ValueError('KINOME_ESM_LIMIT must be nonnegative')
source_hashes = {name: sha256(SOURCE_DIR / name)
                 for name in ['mutant_df.csv', 'wildtype_kinase_data.csv']}
variants = pd.read_csv(SOURCE_DIR / 'mutant_df.csv')
references = pd.read_csv(SOURCE_DIR / 'wildtype_kinase_data.csv')
print('Source rows:', len(variants), 'Reference proteins:', len(references))

## Verify identities, domain selection and coordinates

Full positions and KD bounds are 1-based inclusive. Select KD2 for JAK3 and
RPS6KA5 as in the supplied project source records. Require matching WT residues,
mutated full sequences, WT/mutated KD slices, unique IDs and binary labels.
Invalid records raise an error rather than being skipped.

The supplied table retains 17 outside-KD variants and five identity substitutions.
Outside-KD variants remain unscorable. Identity substitutions are explicitly
flagged historical no-op records, not silently re-curated or called missense.

In [ ]:
validated = validate_variants(variants, references)
assert validated.ID.tolist() == variants.ID.tolist()
assert validated.Experimental.equals(variants.Experimental)
print({'retained': len(validated), 'scorable': int(validated.scorable.sum()),
       'outside_KD': int((validated.scorable == 0).sum()),
       'identity_substitutions': int(validated.identity_substitution.sum())})
validated[['ID', 'position', 'KD start', 'KD end', 'kd_position_1_based',
           'scorable', 'identity_substitution']].head()

## Generate raw KD embeddings (explicit opt-in)

Set `KINOME_RUN_ESM=1` before launching the kernel, or edit `RUN_ESM` above.
The pinned 650M-parameter model may require a large first download; CPU and CUDA
are supported. Use `KINOME_LOCAL_MODEL_ONLY=1` to prohibit model downloads.
`KINOME_ESM_LIMIT` is only for a first-N-row smoke check, not a complete export.

The final ESM layer is 33. Token 0 is BOS; with BOS present the mutant token
index is `full_position - kd_start + 1`. The tokenizer's residue is checked
before extraction. No whole-protein embedding or WT-minus-mutant difference is
used. Checkpoint provenance prevents reuse across different sequences/models.
Failures stop the run, while completed compatible caches allow resumption.

In [ ]:
if RUN_ESM:
    import torch
    torch.manual_seed(18)
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(4)
    selected = validated.iloc[:LIMIT].copy() if LIMIT else validated.copy()
    extractor = ESMExtractor(local_files_only=LOCAL_MODEL_ONLY)
    raw_features = generate_raw_features(selected, extractor, OUTPUT_DIR / 'raw_esm')
    print('Retained export rows:', len(raw_features), 'Embedded:', int(raw_features.scorable.sum()))
    print('Limited smoke export:', bool(LIMIT))
else:
    print('Validation complete. ESM generation is disabled; set KINOME_RUN_ESM=1 to enable it.')

## Outputs and historical-data boundary

Local `outputs/raw_esm/` contains the raw feature CSV, provenance JSON and
resumable per-variant checkpoints. Out-of-KD rows have missing ESM values;
they are not assigned zero vectors. Original input columns and labels remain.

The raw columns are `mutant_first_token_ESM_0..1279` and `mutant_ESM_0..1279`.
This is not a complete model input: family/basic and structural features are
not invented. All original 1,003 training records already have supplied feature
vectors; notebook 2 validates that separate immutable historical matrix.
Generated output is ignored by Git and should not be committed.

In [ ]:
assert all(sha256(SOURCE_DIR / name) == expected for name, expected in source_hashes.items())
print('Both original source CSVs are unchanged.')